## Dependencies

In [ ]:
## libraries
import sys
from pathlib import Path

## path
root = Path.cwd().resolve().parent
sys.path.insert(0, str(root))

## modules
from src.data.builders import load_processed_data
from src.estimators.factories import load_estimators
from src.evaluators.resampling import (
    logo_cross_valid,
    kfold_cross_valid,
    compile_cross_valid,
    results_cross_valid
)
from src.evaluators.config import (
    FEAT_X,
    FEAT_Z, 
    TARGET
)

## Initalization

In [2]:
## setup
data = load_processed_data()
models = load_estimators()

## Training

In [3]:
## leave-one-group-out cross validation (domain)
results_dict_domain = dict()
for name, model in models.items():
    print(f"training {name}...")
    frontier, _ = logo_cross_valid(
        data = data,
        feat_x = FEAT_X,
        feat_z = FEAT_Z,
        estimator_c = model.estimator_c,
        estimator_r = model.estimator_r,
        target = TARGET,
        group = "domain",  ## fold on domain
        n_repeats = 30,
        random_state = 42
    )
    frontier["model"] = name
    results_dict_domain[name] = frontier

training linear_quantile...
training linear_convex...
training linear_laws...
training forest_quantile...
training boosted_quantile...
training xgboost_quantile...
training neural_quantile...
training neural_expectile...
training neural_convex...


In [4]:
## leave-one-group-out cross validation (discipline)
results_dict_disc = dict()
for name, model in models.items():
    print(f"training {name}...")
    frontier, _ = logo_cross_valid(
        data = data,
        feat_x = FEAT_X,
        feat_z = FEAT_Z,
        estimator_c = model.estimator_c,
        estimator_r = model.estimator_r,
        target = TARGET,
        group = "discipline",  ## fold on discipline
        n_repeats = 30,
        random_state = 42
    )
    frontier["model"] = name
    results_dict_disc[name] = frontier

training linear_quantile...
training linear_convex...
training linear_laws...
training forest_quantile...
training boosted_quantile...
training xgboost_quantile...
training neural_quantile...
training neural_expectile...
training neural_convex...


In [5]:
## 2-fold cross validation
results_dict_2fold = dict()
for name, model in models.items():
    print(f"training {name}...")
    frontier, _ = kfold_cross_valid(
        data = data,
        feat_x = FEAT_X,
        feat_z = FEAT_Z,
        estimator_c = model.estimator_c,
        estimator_r = model.estimator_r,
        target = TARGET,
        n_splits = 2,  ## fold on random 50-50 split
        n_repeats = 30,
        random_state = 42
    )
    frontier["model"] = name
    results_dict_2fold[name] = frontier

training linear_quantile...
training linear_convex...
training linear_laws...
training forest_quantile...
training boosted_quantile...
training xgboost_quantile...
training neural_quantile...
training neural_expectile...
training neural_convex...


In [6]:
## 5-fold cross validation
results_dict_5fold = dict()
for name, model in models.items():
    print(f"training {name}...")
    frontier, _ = kfold_cross_valid(
        data = data,
        feat_x = FEAT_X,
        feat_z = FEAT_Z,
        estimator_c = model.estimator_c,
        estimator_r = model.estimator_r,
        target = TARGET,
        n_splits = 5,  ## fold on random 80-20 split
        n_repeats = 30,
        random_state = 42
    )
    frontier["model"] = name
    results_dict_5fold[name] = frontier

training linear_quantile...
training linear_convex...
training linear_laws...
training forest_quantile...
training boosted_quantile...
training xgboost_quantile...
training neural_quantile...
training neural_expectile...
training neural_convex...


In [7]:
## 10-fold cross validation
results_dict_10fold = dict()
for name, model in models.items():
    print(f"training {name}...")
    frontier, _ = kfold_cross_valid(
        data = data,
        feat_x = FEAT_X,
        feat_z = FEAT_Z,
        estimator_c = model.estimator_c,
        estimator_r = model.estimator_r,
        target = TARGET,
        n_splits = 10,  ## fold on random 90-10 split
        n_repeats = 30,
        random_state = 42
    )
    frontier["model"] = name
    results_dict_10fold[name] = frontier

training linear_quantile...
training linear_convex...
training linear_laws...
training forest_quantile...
training boosted_quantile...
training xgboost_quantile...
training neural_quantile...
training neural_expectile...
training neural_convex...


## Post-Processing

In [8]:
## convert all results to dataframes
results_data_domain = compile_cross_valid(results = results_dict_domain)
results_data_disc = compile_cross_valid(results = results_dict_disc)
results_data_2fold = compile_cross_valid(results = results_dict_2fold)
results_data_5fold = compile_cross_valid(results = results_dict_5fold)
results_data_10fold = compile_cross_valid(results = results_dict_10fold)

## Results

In [9]:
## compile transfer bounds and baseline benchmarks
display(
    results_cross_valid(
        {
            "LO-Do-O-CV": results_data_domain,
            "LO-Di-O-CV": results_data_disc,
        },
        {
            "2-Fold-CV": results_data_2fold,
            "5-Fold-CV": results_data_5fold,
            "10-Fold-CV": results_data_10fold,
        },
        keys = [
            "Transfer", 
            "Standard"
        ],
        decimals = 4
    )
)